# Stage B — derive the workspace band

The ablation zeroes directions **across a band of layers**, and light/medium/heavy differ only in how wide that band is. So Control A is undefined until the band is known.

The paper's band (~38%–92% of depth) is Sonnet 4.5's. Qwen3-8B is a different model, and nothing in the released code computes a band. This notebook is that missing step.

**Output is four curves, not a verdict.** The proposal at the end uses arbitrary thresholds; read the curves against it.

**Corpus is WikiText, not the probe-swap prompts.** Deriving the band on the eval you're about to ablate would contaminate it. WikiText also matches what the published lens was fitted on.

**Cost:** a few minutes, 1–2 units.

Set the runtime to **A100 or L4** before cell 1.

## Cell 1 — Setup (no editable install; PYTHONPATH for subprocesses)

In [ ]:
import os, sys, subprocess, torch

!pip -q install -U transformers accelerate huggingface_hub datasets matplotlib

if not os.path.isdir("/content/jacobian-lens"):
    subprocess.run(["git","clone","-q","--depth","1",
                    "https://github.com/anthropics/jacobian-lens.git"],
                   cwd="/content", check=True)

for p in ("/content/ablation", "/content/band"):
    os.makedirs(p, exist_ok=True); open(f"{p}/__init__.py","a").close()
for p in ("/content/jacobian-lens", "/content"):
    if p not in sys.path: sys.path.insert(0, p)
os.environ["PYTHONPATH"] = "/content/jacobian-lens:/content"

import importlib; importlib.invalidate_caches()
import jlens; print("jlens OK:", jlens.__file__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

## Cell 2 — Write `ablation/directions.py`

In [ ]:
%%writefile ablation/directions.py
"""Direction selection and subspace projection for J-space / R-space ablation.

Implements the direction-set half of the ablation harness. Every selector here
produces a set of residual-stream directions of a specified size; the harness
projects them out. Keeping selection separate from projection is what makes the
matched controls of proposal 4.8 cheap: same projection, different selector.

Terminology (guide 2.1): nothing here is "the workspace". These are candidate
directions until Phase 3 says otherwise.
"""

from __future__ import annotations

from dataclasses import dataclass

import torch


# --- J-lens vector construction -------------------------------------------

def lens_vectors(
    unembed_weight: torch.Tensor, jacobian: torch.Tensor, token_ids: torch.Tensor
) -> torch.Tensor:
    """The J-lens vectors for ``token_ids`` at one layer.

    Paper §2.1 defines the J-lens vectors as the rows of ``W_U J_l``. Only the
    requested rows are materialised: the full product is ``[vocab, d_model]``
    and is far too large to hold for a real vocabulary.

    Args:
        unembed_weight: ``W_U``, shape ``[vocab, d_model]``.
        jacobian: ``J_l``, shape ``[d_model, d_model]``.
        token_ids: Shape ``[..., k]``.

    Returns:
        Shape ``[..., k, d_model]``.
    """
    rows = unembed_weight.index_select(0, token_ids.reshape(-1).to(unembed_weight.device))
    rows = rows.to(jacobian.dtype) @ jacobian
    return rows.reshape(*token_ids.shape, jacobian.shape[-1])


# --- Selectors -------------------------------------------------------------

def select_by_rank(
    lens_logits: torch.Tensor,
    k: int,
    *,
    rank_offset: int = 0,
    excluded: torch.Tensor | None = None,
) -> torch.Tensor:
    """Token ids ranked ``rank_offset .. rank_offset + k`` by lens score.

    ``rank_offset=0`` gives the top-k the paper ablates. ``rank_offset=k`` gives
    the next-k, which is the "matched but not selected" control: same lens, same
    size, adjacent rank band. A candidate subspace that matters no more than the
    next-k has not earned H1 (proposal 4.8, extended per the probe-swap design).

    Args:
        lens_logits: Shape ``[n_positions, vocab]``.
        k: Number of directions.
        rank_offset: Rank to start from.
        excluded: Boolean mask ``[n_positions, vocab]``; True entries are never
            selected. This carries the clean-pass exclusion — see
            :func:`clean_top_mask`.

    Returns:
        Shape ``[n_positions, k]``.
    """
    scores = lens_logits.clone()
    if excluded is not None:
        scores = scores.masked_fill(excluded, float("-inf"))
    top = scores.topk(rank_offset + k, dim=-1).indices
    return top[:, rank_offset:]


def random_lens_tokens(
    n_positions: int,
    k: int,
    vocab_size: int,
    generator: torch.Generator,
    *,
    excluded: torch.Tensor | None = None,
    device: torch.device | None = None,
) -> torch.Tensor:
    """Uniformly random token ids — matched-size random control (proposal 4.8).

    Drawn from the lens dictionary rather than isotropically, so the control
    asks "is it *these* lens directions, or any lens directions?". Strictly the
    harder question of the two; run both.
    """
    out = torch.empty(n_positions, k, dtype=torch.long, device=device)
    for p in range(n_positions):
        while True:
            cand = torch.randint(
                vocab_size, (k,), generator=generator, device=generator.device
            ).to(device)
            if excluded is None or not bool(excluded[p, cand].any()):
                out[p] = cand
                break
    return out


def random_isotropic(
    n_positions: int, k: int, d_model: int, generator: torch.Generator,
    *, device: torch.device | None = None, dtype: torch.dtype = torch.float32,
) -> torch.Tensor:
    """Isotropic random unit directions — the paper's random-direction control."""
    v = torch.randn(
        n_positions, k, d_model, generator=generator, device=generator.device,
        dtype=torch.float32,
    ).to(device=device, dtype=dtype)
    return v / v.norm(dim=-1, keepdim=True).clamp_min(1e-12)


def clean_top_mask(
    clean_next_token_logits: torch.Tensor, n_exclude: int = 10
) -> torch.Tensor:
    """Mask marking the clean pass's top-``n_exclude`` predictions per position.

    **This is the confound guard, and it is not optional.** Paper: "we do not
    ablate any tokens that appear in the top-10 tokens of a clean forward pass,
    so as to specifically target the J-space's effects on internal reasoning
    rather than report." Without it, ablation suppresses whatever the model was
    about to say, performance drops for a trivial reason, and H1 gets
    "confirmed" by an artifact.

    Args:
        clean_next_token_logits: Shape ``[n_positions, vocab]`` from an
            unablated forward pass.

    Returns:
        Boolean ``[n_positions, vocab]``, True where a token must not be ablated.
    """
    mask = torch.zeros_like(clean_next_token_logits, dtype=torch.bool)
    top = clean_next_token_logits.topk(n_exclude, dim=-1).indices
    return mask.scatter(-1, top, True)


# --- Projection ------------------------------------------------------------

@dataclass(frozen=True)
class Basis:
    """An orthonormal-row basis with a validity mask for rank-deficient sets."""

    rows: torch.Tensor   # [n_positions, k, d_model], orthonormal rows
    keep: torch.Tensor   # [n_positions, k], float 1/0
    rank: torch.Tensor   # [n_positions], effective rank actually removed


def orthonormalise(vectors: torch.Tensor, *, rtol: float = 1e-6) -> Basis:
    """Orthonormal basis for the span of each position's direction set.

    J-lens vectors are overcomplete and non-orthogonal (paper §2.3), so a set of
    k of them may span fewer than k dimensions. Small singular values are masked
    out rather than dropped, which keeps the operation batched and makes the
    effective rank observable — report it, because "we ablated k directions" is
    false if the span was smaller.
    """
    vectors = vectors.to(torch.float32)
    _, s, vh = torch.linalg.svd(vectors, full_matrices=False)
    keep = (s > rtol * s[..., :1].clamp_min(1e-30)).to(vectors.dtype)
    return Basis(rows=vh, keep=keep, rank=keep.sum(-1))


def project_out(
    hidden: torch.Tensor, basis: Basis, *, mode: str = "subspace"
) -> torch.Tensor:
    """Remove the component of ``hidden`` inside the spanned subspace.

    Args:
        hidden: Shape ``[n_positions, d_model]``.
        mode: ``"subspace"`` projects onto the orthogonal complement of the span
            in one step. ``"sequential"`` removes each direction in turn, which
            is order-dependent for non-orthogonal vectors and therefore removes
            *less* than the full span.

    The paper's phrasing — "zero out the residual stream's projection onto
    each" — does not disambiguate these, and for non-orthogonal J-lens vectors
    they differ. ``"subspace"`` is the default because it is the one that
    actually removes the content; ``"sequential"`` is provided so the choice can
    be tested rather than assumed. Record which was used.
    """
    h = hidden.to(torch.float32)
    if mode == "subspace":
        coeffs = torch.einsum("prd,pd->pr", basis.rows, h) * basis.keep
        return (h - torch.einsum("pr,prd->pd", coeffs, basis.rows)).to(hidden.dtype)
    if mode == "sequential":
        for i in range(basis.rows.shape[1]):
            v = basis.rows[:, i, :] * basis.keep[:, i : i + 1]
            h = h - (h * v).sum(-1, keepdim=True) * v
        return h.to(hidden.dtype)
    raise ValueError(f"unknown mode {mode!r}")

## Cell 3 — Write `ablation/harness.py`

In [ ]:
%%writefile ablation/harness.py
"""Two-pass ablation harness.

Pass 1 is a clean forward pass: it records the residual stream at every band
layer, computes lens logits, and captures the clean next-token distribution.
Pass 2 re-runs with the selected directions projected out.

Two passes are not an optimisation choice — the confound guard of proposal 4.4
(paper: exclude the clean pass's top-10) *requires* knowing the clean output
before choosing what to ablate.

This harness is built to Phase 3 requirements from the first line, per guide
§3a: Control A, Control B, and the Phase 3 sweep all run through it unchanged.
"""

from __future__ import annotations

from dataclasses import dataclass, field, asdict
from typing import Any, Literal, Sequence

import torch

from jlens.hooks import ActivationRecorder
from jlens.lens import JacobianLens

from .directions import (
    Basis,
    clean_top_mask,
    lens_vectors,
    orthonormalise,
    project_out,
    random_isotropic,
    random_lens_tokens,
    select_by_rank,
)

Selector = Literal["topk", "next_k", "random_lens", "random_iso", "none"]


def record_at_or(spec, final):
    return sorted({*spec.layers, final})


def _mask_from_ids(ids: torch.Tensor, shape) -> torch.Tensor:
    """Rebuild the clean-top-k boolean mask from cached ids."""
    m = torch.zeros(shape, dtype=torch.bool, device=ids.device)
    return m.scatter(-1, ids, True)


@dataclass(frozen=True)
class AblationSpec:
    """One fully-specified ablation condition. Serialise this into every result.

    Attributes:
        layers: Band of block indices to ablate at. Light/medium/heavy differ
            only here — the paper varies the layer range, not k.
        k: Directions removed per position (proposal 4.7 sweeps this; the paper
            fixed it at 10). Sweeping k *and* layers multiplies runs — state
            which axis in prereg_phase3.md.
        selector: Which directions. ``"none"`` is the clean baseline.
        seed: Required for every random selector (guide §1.2).
        exclude_clean_top: Confound guard size. **Do not set to 0** except as a
            deliberate, logged demonstration of the artifact it prevents.
        mode: Projection mode; see :func:`project_out`.
        positions: Token positions to ablate at; ``None`` means all.
    """

    layers: tuple[int, ...]
    k: int
    selector: Selector = "topk"
    seed: int | None = None
    exclude_clean_top: int = 10
    mode: str = "subspace"
    positions: tuple[int, ...] | None = None

    def __post_init__(self) -> None:
        if self.selector in ("random_lens", "random_iso") and self.seed is None:
            raise ValueError(
                "random selectors require an explicit seed — an unseeded "
                "matched-random baseline is not reproducible and proposal 4.8 "
                "results computed against it are not reportable"
            )

    def key(self) -> str:
        import hashlib, json
        blob = json.dumps(asdict(self), sort_keys=True, default=str)
        return hashlib.sha256(blob.encode()).hexdigest()[:16]


@dataclass
class AblationResult:
    logits: torch.Tensor              # [n_positions, vocab] ablated next-token logits
    clean_logits: torch.Tensor        # [n_positions, vocab] unablated
    effective_rank: dict[int, torch.Tensor] = field(default_factory=dict)
    spec: AblationSpec | None = None


def prepare_lens(lens: JacobianLens, device) -> JacobianLens:
    """Move the Jacobians onto the compute device. **Does not touch dtype.**

    ``JacobianLens.__init__`` does ``J.float()`` on every Jacobian, so the class
    holds float32 regardless of what was on disk (``save`` writes fp16 purely
    for compactness). ``apply()`` correspondingly casts residuals with
    ``.float()`` before ``transport``. The library's internal contract is
    float32 throughout, and ``HFLensModel.unembed`` casts to the head's dtype
    itself, so nothing downstream needs the model's dtype here.

    An earlier version of this function cast the Jacobians to the model's dtype.
    That broke the contract and produced
    ``RuntimeError: expected mat1 and mat2 to have the same dtype`` inside
    ``transport``. Callers passing activations straight from
    ``ActivationRecorder`` (which are in the *model's* dtype, not float32) must
    cast those to float — see :func:`build_cache`.

    Only the device move is needed: without it ``transport`` copies a
    ``[d_model, d_model]`` matrix host-to-device on every call.
    """
    lens.jacobians = {k: v.to(device=device) for k, v in lens.jacobians.items()}
    return lens


@dataclass
class PromptCache:
    """Per-prompt work that every condition would otherwise repeat.

    The clean forward pass and the lens readout are identical across all
    conditions for a given prompt — only the direction *selection* differs. A
    37-condition sweep without this recomputes both 37 times.

    What is cached is deliberately small: the ranked token ids per layer, not
    the lens logits themselves. Lens logits are ``[n_positions, vocab]``, which
    at a 150k vocabulary is megabytes per layer per prompt; the ranked ids are
    ``[n_positions, k_max]``. Any ``k <= k_max`` is then a slice.

    ``k_max`` must be at least ``2 * max(k)`` in the sweep, because the
    ``next_k`` selector reads ranks ``k..2k``.
    """

    ids: torch.Tensor
    n_pos: int
    clean_logits: torch.Tensor
    excluded_ids: torch.Tensor | None          # [n_pos, n_exclude]
    ranked_ids: dict[int, torch.Tensor]        # layer -> [n_pos, k_max]
    k_max: int


@torch.no_grad()
def build_cache(
    model: Any, lens: JacobianLens, prompt: str, layers: Sequence[int],
    *, k_max: int, exclude_clean_top: int = 10, max_seq_len: int = 512,
) -> PromptCache:
    """Run the clean pass once and rank directions once, for reuse."""
    ids = model.encode(prompt, max_length=max_seq_len)
    final = model.n_layers - 1
    record_at = sorted({*layers, final})

    with ActivationRecorder(model.layers, record_at) as rec:
        model.forward(ids)
        acts = {i: rec.activations[i][0].detach() for i in record_at}
    clean_logits = model.unembed(acts[final])

    excluded = (clean_top_mask(clean_logits, exclude_clean_top)
                if exclude_clean_top > 0 else None)
    excluded_ids = (clean_logits.topk(exclude_clean_top, dim=-1).indices
                    if exclude_clean_top > 0 else None)

    ranked = {}
    for layer in layers:
        # .float() to match the lens's float32 Jacobians, as apply() does.
        lens_logits = model.unembed(lens.transport(acts[layer].float(), layer))
        ranked[layer] = select_by_rank(lens_logits, k_max, excluded=excluded)

    return PromptCache(ids, ids.shape[1], clean_logits, excluded_ids, ranked, k_max)


class _Ablator:
    """Forward hooks that project out precomputed per-position direction sets."""

    def __init__(
        self,
        blocks: Sequence[torch.nn.Module],
        bases: dict[int, Basis],
        position_mask: torch.Tensor | None,
        mode: str,
    ) -> None:
        self._blocks, self._bases = blocks, bases
        self._position_mask, self._mode = position_mask, mode
        self._handles: list[Any] = []

    def _hook(self, index: int):
        basis = self._bases[index]

        def fn(module, inputs, output):
            is_tuple = not torch.is_tensor(output)
            tensor = output[0] if is_tuple else output
            # tensor: [batch, seq, d_model]; harness runs batch=1.
            h = tensor[0]
            new = project_out(h, basis, mode=self._mode)
            if self._position_mask is not None:
                new = torch.where(self._position_mask[:, None], new, h)
            tensor = torch.cat([new[None], tensor[1:]], dim=0)
            return (tensor, *output[1:]) if is_tuple else tensor

        return fn

    def __enter__(self):
        try:
            for i in self._bases:
                self._handles.append(self._blocks[i].register_forward_hook(self._hook(i)))
        except Exception:
            self.__exit__()
            raise
        return self

    def __exit__(self, *exc) -> None:
        for h in self._handles:
            h.remove()
        self._handles = []


@torch.no_grad()
def run_ablation(
    model: Any,
    lens: JacobianLens,
    unembed_weight: torch.Tensor,
    prompt: str,
    spec: AblationSpec,
    *,
    max_seq_len: int = 512,
    cache: PromptCache | None = None,
) -> AblationResult:
    """Run one ablation condition end to end.

    Args:
        model: Anything satisfying ``jlens.protocol.LensModel``.
        lens: Fitted lens. ``spec.layers`` must be a subset of its source layers
            for lens-based selectors.
        unembed_weight: ``W_U``, ``[vocab, d_model]`` — usually
            ``model.lm_head.weight``. Passed explicitly because the LensModel
            protocol exposes ``unembed()`` (norm + head) but not ``W_U`` itself.
    """
    final = model.n_layers - 1

    # --- Pass 1: clean (skipped entirely when a cache is supplied) ---
    if cache is not None:
        if spec.k * (2 if spec.selector == "next_k" else 1) > cache.k_max:
            raise ValueError(
                f"cache holds k_max={cache.k_max} ranked directions but this "
                f"condition needs {spec.k * (2 if spec.selector == 'next_k' else 1)}. "
                "Rebuild the cache with a larger k_max."
            )
        ids, n_pos = cache.ids, cache.n_pos
        clean_logits, acts = cache.clean_logits, None
    else:
        ids = model.encode(prompt, max_length=max_seq_len)
        n_pos = ids.shape[1]
        with ActivationRecorder(model.layers, sorted({*spec.layers, final})) as rec:
            model.forward(ids)
            acts = {i: rec.activations[i][0].detach() for i in record_at_or(spec, final)}
        clean_logits = model.unembed(acts[final])

    if spec.selector == "none":
        return AblationResult(clean_logits, clean_logits, spec=spec)

    excluded = None
    if spec.exclude_clean_top > 0:
        excluded = (
            _mask_from_ids(cache.excluded_ids, clean_logits.shape)
            if cache is not None
            else clean_top_mask(clean_logits, spec.exclude_clean_top)
        )

    # --- Direction selection, per band layer ---
    gen = torch.Generator(device="cpu")
    if spec.seed is not None:
        gen.manual_seed(spec.seed)
    bases: dict[int, Basis] = {}
    ref = clean_logits
    for layer in spec.layers:
        h = acts[layer] if acts is not None else ref
        if spec.selector == "random_iso":
            vecs = random_isotropic(
                n_pos, spec.k, model.d_model, gen, device=h.device, dtype=h.dtype
            )
        else:
            ranked = cache.ranked_ids[layer] if cache is not None else None
            if ranked is None:
                lens_logits = model.unembed(lens.transport(h.float(), layer))
            if spec.selector == "topk":
                tok = (ranked[:, : spec.k] if ranked is not None
                       else select_by_rank(lens_logits, spec.k, excluded=excluded))
            elif spec.selector == "next_k":
                tok = (ranked[:, spec.k : 2 * spec.k] if ranked is not None
                       else select_by_rank(lens_logits, spec.k,
                                           rank_offset=spec.k, excluded=excluded))
            elif spec.selector == "random_lens":
                tok = random_lens_tokens(
                    n_pos, spec.k, clean_logits.shape[-1], gen,
                    excluded=excluded, device=clean_logits.device,
                )
            else:
                raise ValueError(f"unknown selector {spec.selector!r}")
            vecs = lens_vectors(unembed_weight, lens.jacobians[layer].to(h.device), tok)
        bases[layer] = orthonormalise(vecs)

    pos_mask = None
    if spec.positions is not None:
        pos_mask = torch.zeros(n_pos, dtype=torch.bool, device=ids.device)
        pos_mask[list(spec.positions)] = True

    # --- Pass 2: ablated ---
    with _Ablator(model.layers, bases, pos_mask, spec.mode):
        with ActivationRecorder(model.layers, [final]) as rec2:
            model.forward(ids)
            ablated_final = rec2.activations[final][0].detach()

    return AblationResult(
        logits=model.unembed(ablated_final),
        clean_logits=clean_logits,
        effective_rank={l: b.rank for l, b in bases.items()},
        spec=spec,
    )


def greedy_match(result: AblationResult, answer_id: int, position: int = -1) -> dict:
    """Score one prompt: did the greedy next token match, clean and ablated?

    The Control A metric per DECISION_control_A §4.4 — greedy next-token
    accuracy against probe-swap.json's ``answer`` field.
    """
    return {
        "clean_correct": int(result.clean_logits[position].argmax()) == answer_id,
        "ablated_correct": int(result.logits[position].argmax()) == answer_id,
    }

## Cell 4 — Write `band/derive.py`

In [ ]:
%%writefile band/derive.py
"""Workspace-band derivation.

The paper's band (~L38-92 on a reindexed 0-100 scale) is Sonnet 4.5's. Every
ablation experiment reports over the band, so it must be derived per model
before ablation can run. Nothing in the released code computes it.

Four J-lens-derived statistics, per the paper:

  kurtosis    excess kurtosis of the readout logit distribution. ~0 through the
              early block, rises from ~1/3 depth. MARKS THE START.
  topk_acc    top-k accuracy of the lens at predicting the model's actual next
              token. ~0 early, ticks up at band start, jumps steeply in the
              final layers. MARKS THE END (motor onset).
  autocorr    autocorrelation of the top-1 lens token across nearby positions,
              against a position-shuffled null. Persistence of abstract content.
  eff_dim     effective linear dimensionality of ``W_U J_l``. Small early, rises
              sharply at band onset, again at the motor transition.

CAVEAT, stated by the paper itself: all four derive from the J-lens, so layer
effects could be artifacts of the method rather than facts about the model. The
paper answers this with the ignition experiment, which uses no lens at all. That
is not implemented here. For Phase 0 (a language model, replicating a published
band) these four are adequate; for Phase 2 in the recommender domain they are
NOT, and a readout-independent check is required there.

Only kurtosis and eff_dim port to a recommender without reinterpretation.
topk_acc and autocorr both presuppose a token stream with language-like local
redundancy.
"""

from __future__ import annotations

from dataclasses import dataclass

import torch


@dataclass
class LayerStats:
    layer: int
    kurtosis: float
    topk_acc: float
    autocorr: float
    eff_dim: float


def excess_kurtosis(logits: torch.Tensor) -> float:
    """Excess kurtosis of the readout distribution, averaged over positions.

    Computed per position then averaged, not pooled: pooling would mix
    across-position variance into the shape statistic.
    """
    x = logits.float()
    c = x - x.mean(-1, keepdim=True)
    var = c.pow(2).mean(-1)
    k = c.pow(4).mean(-1) / var.pow(2).clamp_min(1e-12) - 3.0
    return float(k.mean())


def topk_accuracy(
    lens_logits: torch.Tensor, model_logits: torch.Tensor, k: int = 5
) -> float:
    """Fraction of positions where the model's argmax is in the lens's top k.

    Scored against the model's OWN next-token prediction, not ground truth —
    this measures how far the lens has converged onto the output, which is what
    marks the motor transition.
    """
    target = model_logits.argmax(-1)
    topk = lens_logits.topk(k, dim=-1).indices
    return float((topk == target[:, None]).any(-1).float().mean())


def top1_autocorrelation(
    lens_logits: torch.Tensor, *, lag: int = 1, n_shuffles: int = 20,
    generator: torch.Generator | None = None,
) -> float:
    """Excess agreement of the top-1 lens token at distance ``lag``, over a
    position-shuffled null.

    Returned as observed minus null, so 0 means "no more persistent than
    chance". Reporting the raw rate instead would confound persistence with a
    single token dominating every position.
    """
    top1 = lens_logits.argmax(-1)
    if top1.numel() <= lag:
        return 0.0
    observed = float((top1[:-lag] == top1[lag:]).float().mean())
    null = 0.0
    for _ in range(n_shuffles):
        perm = torch.randperm(top1.numel(), generator=generator)
        s = top1[perm]
        null += float((s[:-lag] == s[lag:]).float().mean())
    return observed - null / n_shuffles


def effective_dimensionality(
    unembed_weight: torch.Tensor, jacobian: torch.Tensor, *, n_rows: int = 4096,
    generator: torch.Generator | None = None,
) -> float:
    """Participation ratio of the singular values of ``W_U J_l``.

    ``(sum s^2)^2 / sum s^4`` — 1.0 if one direction dominates, rank if all are
    equal. Estimated on a random row subsample because the full product is
    ``[vocab, d_model]`` and materialising it is not affordable; the subsample
    size is recorded since the estimate depends on it.
    """
    v = unembed_weight.shape[0]
    idx = (torch.randperm(v, generator=generator)[:n_rows]
           if v > n_rows else torch.arange(v))
    m = unembed_weight[idx].to(jacobian.dtype) @ jacobian
    s = torch.linalg.svdvals(m.float())
    s2 = s.pow(2)
    # Participation ratio: (sum s^2)^2 / sum s^4. Normalised by the largest
    # singular value first — the raw fourth power overflows float32 on real
    # weight matrices, which silently returns 0 rather than raising.
    s2 = s2 / s2.max().clamp_min(1e-30)
    return float(s2.sum().pow(2) / s2.pow(2).sum().clamp_min(1e-30))


def layer_stats(
    lens, model, prompts, unembed_weight, *, layers=None, k: int = 5,
    skip_first: int = 4, max_seq_len: int = 128, seed: int = 0,
) -> list[LayerStats]:
    """Compute all four statistics per layer, averaged over prompts."""
    layers = list(lens.source_layers if layers is None else layers)
    gen = torch.Generator().manual_seed(seed)
    acc = {l: [[], [], []] for l in layers}

    for prompt in prompts:
        lens_logits, model_logits, _ = lens.apply(
            model, prompt, layers=layers, max_seq_len=max_seq_len
        )
        for l in layers:
            ll = lens_logits[l][skip_first:]
            ml = model_logits[skip_first:]
            if ll.shape[0] < 2:
                continue
            acc[l][0].append(excess_kurtosis(ll))
            acc[l][1].append(topk_accuracy(ll, ml, k=k))
            acc[l][2].append(top1_autocorrelation(ll, generator=gen))

    out = []
    for l in layers:
        ed = effective_dimensionality(
            unembed_weight, lens.jacobians[l].to(unembed_weight.device), generator=gen
        )
        m = lambda xs: sum(xs) / len(xs) if xs else float("nan")
        out.append(LayerStats(l, m(acc[l][0]), m(acc[l][1]), m(acc[l][2]), ed))
    return out


def propose_band(stats: list[LayerStats], *, kurt_frac: float = 0.25,
                 acc_frac: float = 0.50) -> tuple[int, int]:
    """A FIRST PASS at the band. Not a substitute for reading the curves.

    Start: first layer whose kurtosis exceeds ``kurt_frac`` of the maximum.
    End:   last layer before top-k accuracy exceeds ``acc_frac`` of the maximum
           (the motor transition, where the lens collapses onto the output).

    The thresholds are arbitrary and are exposed so they can be recorded rather
    than buried. Plot the curves and check the proposal against them; if the
    statistics disagree about where the band is, that disagreement is a finding
    and the band should not be forced.
    """
    ks = [s.kurtosis for s in stats]
    accs = [s.topk_acc for s in stats]
    kt, at = kurt_frac * max(ks), acc_frac * max(accs)
    start = next((s.layer for s, v in zip(stats, ks) if v > kt), stats[0].layer)
    end = stats[-1].layer
    for s, v in zip(stats, accs):
        if s.layer > start and v > at:
            end = s.layer
            break
    return start, end

## Cell 5 — Write `derive_band.py`

In [ ]:
%%writefile derive_band.py
"""Stage B — derive the workspace band for this model.

The paper's band (~L38-92 on a reindexed 0-100 scale) is Sonnet 4.5's. Every
ablation experiment reports over the band, and light/medium/heavy differ only
in its width. Nothing in the released code computes one, and no step for
deriving it exists in the proposal or the execution guide. This is that step.

Corpus: WikiText-103, via jlens' own loader. NOT the probe-swap prompts —
deriving the band on the eval you are about to ablate would contaminate it.
WikiText also matches what the published lens was fitted on.

Output is FOUR CURVES, not a verdict. `propose_band` gives a first pass with
explicit, arbitrary thresholds; read the curves against it.

Usage:
    python derive_band.py --model Qwen/Qwen3-8B \
        --lens-file qwen3-8b/jlens/Salesforce-wikitext/Qwen3-8B_jacobian_lens.pt \
        --out results/raw/band_qwen3-8b/
"""
from __future__ import annotations
import argparse, json, time
from dataclasses import asdict

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import jlens
from jlens.examples import load_wikitext_prompts
from jlens.lens import JacobianLens
from ablation.harness import prepare_lens
from band.derive import layer_stats, propose_band

LENS_REPO = "neuronpedia/jacobian-lens"


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--lens-file", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--dtype", default="bfloat16")
    ap.add_argument("--n-prompts", type=int, default=20)
    ap.add_argument("--topk", type=int, default=5)
    ap.add_argument("--skip-first", type=int, default=4,
                    help="positions dropped from each statistic. Low, per A.7: "
                         "position masking yielded no meaningful improvement, so "
                         "there is no reason to inherit the code default of 16")
    ap.add_argument("--max-seq-len", type=int, default=128)
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = getattr(torch, args.dtype)

    print(f"loading {args.model} ...")
    hf = AutoModelForCausalLM.from_pretrained(args.model, dtype=dtype, device_map=device)
    tok = AutoTokenizer.from_pretrained(args.model)
    lm = jlens.from_hf(hf, tok)
    print(f"  n_layers={lm.n_layers} d_model={lm.d_model}")

    lens = JacobianLens.from_pretrained(LENS_REPO, filename=args.lens_file)
    lens = prepare_lens(lens, device)   # device only; the class holds float32 by design
    print(f"  lens source_layers {min(lens.source_layers)}..{max(lens.source_layers)}")

    prompts = load_wikitext_prompts(args.n_prompts)
    print(f"  {len(prompts)} WikiText passages")

    wu = lm._lm_head.weight.detach()
    t0 = time.perf_counter()
    stats = layer_stats(lens, lm, prompts, wu, k=args.topk,
                        skip_first=args.skip_first, max_seq_len=args.max_seq_len,
                        seed=args.seed)
    print(f"  computed in {time.perf_counter()-t0:.0f}s\n")

    print(f"{'layer':>5} {'depth%':>7} {'kurtosis':>10} {'topk_acc':>9} {'autocorr':>9} {'eff_dim':>8}")
    for s in stats:
        print(f"{s.layer:>5} {100*s.layer/(lm.n_layers-1):>6.0f}% {s.kurtosis:>10.3f} "
              f"{s.topk_acc:>9.3f} {s.autocorr:>9.3f} {s.eff_dim:>8.2f}")

    start, end = propose_band(stats)
    d = lm.n_layers - 1
    print(f"\nPROPOSED BAND: layers {start}..{end}  "
          f"({100*start/d:.0f}%-{100*end/d:.0f}% of depth)")
    print(f"  paper's band for Sonnet 4.5 was ~38%-92% of depth, for reference only")
    print("\n  Thresholds used: kurtosis > 25% of peak (start), "
          "topk_acc > 50% of peak (end).")
    print("  These are ARBITRARY. Read the curves before accepting the proposal.")

    import pathlib
    out = pathlib.Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    (out / "band_stats.json").write_text(json.dumps({
        "model": args.model, "n_layers": lm.n_layers, "d_model": lm.d_model,
        "lens_file": args.lens_file, "config": vars(args),
        "stats": [asdict(s) for s in stats],
        "proposed_band": {"start": start, "end": end,
                          "kurt_frac": 0.25, "acc_frac": 0.50},
    }, indent=2))
    print(f"\nwrote {out/'band_stats.json'}")


if __name__ == "__main__":
    main()

## Cell 6 — Derive the band

Loads the model and the published lens, runs four statistics across all 35 lens layers on 20 WikiText passages.

`--skip-first 4` is deliberate: §A.7 reports that position masking gave no meaningful improvement, so there is no reason to inherit the code's default of 16.

In [ ]:
!python derive_band.py \
    --model Qwen/Qwen3-8B \
    --lens-file qwen3-8b/jlens/Salesforce-wikitext/Qwen3-8B_jacobian_lens.pt \
    --out results/raw/band_qwen3-8b/ \
    --dtype bfloat16 \
    --n-prompts 20 \
    --skip-first 4

## Cell 7 — Plot the curves

**This is the actual deliverable.** The printed proposal is a first pass; the curves are what you read.

What to look for, per the paper:

- **kurtosis** — flat through the early block, rises from roughly a third of depth. Marks the **start**.
- **topk_acc** — near zero early, ticks up at band start, then jumps steeply in the final layers. That jump is the motor transition and marks the **end**.
- **autocorr** — near zero early, rises, peaks mid-band, falls late.
- **eff_dim** — small early, rises sharply at onset, rises again at the motor transition.

In [ ]:
import json
import matplotlib.pyplot as plt

d = json.load(open("results/raw/band_qwen3-8b/band_stats.json"))
S = d["stats"]; n = d["n_layers"]
x = [s["layer"] for s in S]
depth = [100 * s["layer"] / (n - 1) for s in S]
b = d["proposed_band"]

fig, ax = plt.subplots(4, 1, figsize=(9, 11), sharex=True)
for a, key, title in zip(ax,
        ["kurtosis", "topk_acc", "autocorr", "eff_dim"],
        ["Excess kurtosis  (marks band START)",
         "Top-k accuracy vs model argmax  (marks band END / motor onset)",
         "Top-1 autocorrelation over shuffled null",
         "Effective dimensionality of W_U J_l"]):
    a.plot(x, [s[key] for s in S], marker="o", ms=3)
    a.axvspan(b["start"], b["end"], alpha=0.12, color="tab:green")
    a.set_title(title, fontsize=10, loc="left")
    a.grid(alpha=0.3)
ax[-1].set_xlabel("layer")
fig.suptitle(f"{d['model']} — proposed band {b['start']}..{b['end']} "
             f"({100*b['start']/(n-1):.0f}%-{100*b['end']/(n-1):.0f}% depth); "
             f"paper's was ~38%-92%", fontsize=11)
fig.tight_layout()
plt.savefig("results/raw/band_qwen3-8b/band_curves.png", dpi=130)
plt.show()

print("Do the four statistics agree on where the band is?")
print("If they disagree, that disagreement is a FINDING — do not average it away.")

## Cell 8 — Download

In [ ]:
from google.colab import files
files.download("results/raw/band_qwen3-8b/band_stats.json")
files.download("results/raw/band_qwen3-8b/band_curves.png")

---
## Report back

Paste the cell 6 table (all 35 rows), the proposed band, and describe what the curves look like — or attach the PNG.

**Do not treat the proposed band as settled until the curves are read.** The band width feeds straight into Stage C's cost and into what the three ablation strengths mean.

### Caveat to carry forward

All four statistics derive from the J-lens, so a layer effect could be an artifact of the method rather than a fact about the model. The paper answers this with the ignition experiment, which uses no lens at all — not implemented here. For Phase 0, replicating a published band on a language model, these four are adequate. **For Phase 2 in the recommender domain they will not be.**